In [ ]:
"""Author: Abrahim Mejía Valdez - Software Engineer"""

import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv()

endpoint = os.getenv("AZURE_FOUNDRY_ENDPOINT")
model_name = "gpt-5-nano"
api_key = os.getenv("AZURE_FOUNDRY_API_KEY")
max_tokens = 2500

if not endpoint or not api_key:
    raise EnvironmentError("Missing AZURE_FOUNDRY_ENDPOINT or AZURE_FOUNDRY_API_KEY")

try:
    # Create the client
    client = OpenAI(base_url=endpoint, api_key=api_key)

    # Setting up instructions for the agent and first prompt
    messages = [
        {
            "role": "system",
            "content": "You are an AI engineer with expertise in developing and maintaining enterprise-grade applications.",
        },
        {
            "role": "user",
            "content": "Pick a business area that might be worth exploring for an Agentic AI opportunity. Generate text only output with a max of 100 chars.",
        },
    ]

    # Then first call
    response = client.chat.completions.create(
        model=model_name, messages=messages, max_completion_tokens=max_tokens
    )

    # Then read the business idea
    business_area = response.choices[0].message.content
    print(f"Business area: {business_area}")

    # Include the business area within the message as well as the previous messages

    messages.extend(
        [
            {"role": "assistant", "content": business_area},
            {
                "role": "user",
                "content": "Present a pain-point in that industry - something challenging that might be ripe for an Agentic solution. No more than 200 chars.",
            },
        ]
    )

    # Request the agent for the pain-point
    response = client.chat.completions.create(
        model=model_name, messages=messages, max_completion_tokens=max_tokens
    )

    challenge = response.choices[0].message.content
    print(f"Challenge: {challenge}")

    messages.extend(
        [
            {"role": "assistant", "content": challenge},
            {
                "role": "user",
                "content": """
                    Propose the Agentic AI solution with the following rules:
                    - Use markdown format
                    - 500 chars at most
                    - It MUST have a title section, an introduction, the core idea and why it is a good Agentic solution
                    """,
            },
        ]
    )
    # Request the agent for the solution
    response = client.chat.completions.create(model=model_name, messages=messages)
    solution = response.choices[0].message.content

    print("Solution:\n" + solution)
    display(Markdown(solution))
except Exception as ex:
    print(f"An error has occurred:\n {ex}")